# Decision 3 — Capital Adequacy under Vendor Uncertainty

**Mechanism.** Physical flood damage raises borrower PD and LGD, increasing portfolio expected loss (EL). If EL is charged against capital, the bank's CET1 ratio falls. Vendor disagreement on damage ratios translates directly into a distribution of possible CET1 outcomes — some above the regulatory minimum, some below. The capital threshold creates a binary cliff: a small change in which vendor the bank consults can flip the assessment from "adequately capitalised" to "in breach". This notebook quantifies that exposure using the same CFRF/GARP portfolio as Decisions 1 and 2.

**Portfolio consistency.** Setting `PORTFOLIO_SEED = 42` reproduces the same 100-loan book used in D1 and D2. The reduced-form PD/LGD transmission parameters (`ALPHA`, `LAMBDA`) are shared with D1. RWA is computed from the Basel IRB Vasicek formula applied to baseline PD0, LGD0, and per-loan maturity.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm as _norm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

BLUE, RED, GREY, AMBER = '#4878CF', '#E84D0E', '#888888', '#F0A500'

In [ ]:
# =============================================================================
# PARAMETERS -- edit this cell to explore how results change
# =============================================================================

# --- Seeds (match Decision 1 for portfolio consistency) ---
PORTFOLIO_SEED = 42
VENDOR_SEED    = 1       # single-vendor display cells
N_DRAWS        = 2000    # vendor uncertainty simulation

# --- Probability framing ---
# 'conditional'   : flood treated as certain (p_flood = 1.0)
# 'unconditional' : probability-weighted by 200yr RP (p_flood = 1/200)
FRAMING        = 'conditional'
P_FLOOD_ANNUAL = 1 / 200   # used only when FRAMING == 'unconditional'

# --- Reduced-form transmission parameters ---
ALPHA  = 2.0    # PD sensitivity:  PD_v  = PD_0  * exp(alpha  * d_v)
LAMBDA = 0.25   # LGD sensitivity: LGD_v = min(LGD_0 + lambda * d_v, 1.0)

# --- Reduced-form baseline portfolio parameters (must match Decision 1) ---
EAD_LOW,   EAD_HIGH   = 0.5,    5.0    # GBPm, Uniform
PD0_MU,    PD0_SIGMA  = -5.116, 1.0   # LogNormal; median ~0.6%
LGD0_LOW,  LGD0_HIGH  = 0.10,   0.30  # Uniform
MAT_LOW,   MAT_HIGH   = 3,      15    # years, Uniform int

# --- Capital adequacy parameters ---
CET1_THRESHOLD = 0.045   # regulatory minimum CET1 ratio; adjust freely
CET1_START     = 0.120   # bank's initial CET1 ratio (well above threshold)
MATURITY_IRB   = 2.5     # years, Basel maturity adjustment input (fixed for all loans)

# --- Pairwise vendor comparison ---
VENDOR_A = 1
VENDOR_B = 2

# --- Data ---
DATA_PATH = '../../data/raw/cfrf_garp_defended_flood.csv'

In [ ]:
# =============================================================================
# DATA, PORTFOLIO, AND CAPITAL CALIBRATION
# Run this cell after editing parameters above.
# =============================================================================

# ── Load CFRF/GARP flood data ─────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df.rename(columns={'minimim_dr': 'minimum_dr'}, inplace=True)
df = df.dropna(subset=['property_rank']).reset_index(drop=True)
df['property_rank'] = df['property_rank'].astype(int)
n = len(df)

# Triangular distribution parameters (mode from mean constraint: c = 3μ − min − max)
tri_a = df['minimum_dr'].values.astype(float)
tri_b = df['maximum_dr'].values.astype(float)
tri_c = np.clip(3.0 * df['mean_dr'].values - tri_a - tri_b, tri_a, tri_b)

def triangular_sample(rng, a, b, c):
    """Draw one sample per property from Triangular(a, c, b) via inverse CDF."""
    u     = rng.uniform(size=len(a))
    span  = np.where(b > a, b - a, 1.0)
    fc    = np.where(b > a, (c - a) / span, 0.0)
    left  = a + np.sqrt(np.maximum(u,       0) * span * np.maximum(c - a, 0))
    right = b - np.sqrt(np.maximum(1 - u,   0) * span * np.maximum(b - c, 0))
    x     = np.where(u <= fc, left, right)
    return np.where(b > a, np.clip(x, 0, 1), a)

# ── Sample one vendor draw ────────────────────────────────────────────────────
vendor_rng = np.random.default_rng(VENDOR_SEED)
d_vendor   = triangular_sample(vendor_rng, tri_a, tri_b, tri_c)

# ── Loan portfolio ────────────────────────────────────────────────────────────
port_rng = np.random.default_rng(PORTFOLIO_SEED)
ead  = port_rng.uniform(EAD_LOW, EAD_HIGH, size=n)
pd0  = np.clip(
    stats.lognorm.rvs(s=PD0_SIGMA, scale=np.exp(PD0_MU), size=n,
                      random_state=int(port_rng.integers(2**31))),
    1e-6, 0.9999
)
lgd0 = port_rng.uniform(LGD0_LOW, LGD0_HIGH, size=n)
mat  = port_rng.integers(MAT_LOW, MAT_HIGH + 1, size=n).astype(float)
el0  = ead * pd0 * lgd0   # baseline expected loss per loan

# ── IRB Vasicek capital formula (Basel corporate, non-SME) ────────────────────
def irb_rwa(pd, lgd, ead_arr, m):
    """
    Per-loan RWA from the Basel II/III IRB formula for corporate exposures.
    Accepts arrays; m is scalar (MATURITY_IRB) or array (per-loan maturity).
    """
    pd = np.clip(pd, 0.0003, 0.9999)   # floor at 0.03% per Basel text
    # Asset correlation: decreasing function of PD (diversification argument)
    exp50 = np.exp(-50.0)              # ≈ 0; pre-compute for clarity
    f_pd  = (1.0 - np.exp(-50.0 * pd)) / (1.0 - exp50)
    R     = 0.12 * f_pd + 0.24 * (1.0 - f_pd)
    # Maturity adjustment
    b  = (0.11852 - 0.05478 * np.log(pd)) ** 2
    ma = (1.0 + (m - 2.5) * b) / (1.0 - 1.5 * b)
    ma = np.maximum(ma, 0.0)           # guard against negative denominator edge cases
    # Capital requirement per unit EAD
    K = (lgd * _norm.cdf(
            _norm.ppf(pd) / np.sqrt(1.0 - R)
            + np.sqrt(R / (1.0 - R)) * _norm.ppf(0.999)
         ) - lgd * pd) * ma
    K = np.maximum(K, 0.0)
    return 12.5 * K * ead_arr

rwa_loans = irb_rwa(pd0, lgd0, ead, MATURITY_IRB)
RWA_0     = rwa_loans.sum()

# Fallback if RWA is implausibly small (degenerate inputs)
if RWA_0 < 1e-6 * ead.sum():
    print("WARNING: IRB RWA near zero — using fallback RWA_0 = 0.5 * total EAD")
    RWA_0 = 0.5 * ead.sum()
    rwa_loans = np.full(n, RWA_0 / n)

Capital_0 = CET1_START * RWA_0
EL_0      = el0.sum()
buffer_bps = (CET1_START - CET1_THRESHOLD) * 1e4   # buffer in basis points of RWA
max_loss_before_breach = (CET1_START - CET1_THRESHOLD) * RWA_0

# ── Sort index (shared by figure cells) ──────────────────────────────────────
idx   = np.argsort(df['mean_dr'].values)
x_pos = np.arange(n)

print("=== Capital calibration ===")
print(f"  Total EAD:            GBP{ead.sum():.2f}m")
print(f"  Baseline RWA:         GBP{RWA_0:.2f}m  ({RWA_0/ead.sum()*100:.1f}% of EAD)")
print(f"  Capital_0  (CET1 × RWA): GBP{Capital_0:.3f}m")
print(f"  Baseline CET1:        {CET1_START*100:.1f}%")
print(f"  CET1 threshold:       {CET1_THRESHOLD*100:.1f}%")
print(f"  Buffer above threshold: {buffer_bps:.0f} bps")
print(f"  Max loss before breach: GBP{max_loss_before_breach:.3f}m")
print(f"  Baseline EL:          GBP{EL_0:.4f}m  ({EL_0/max_loss_before_breach*100:.1f}% of buffer)")

In [ ]:
# Standard opening figure: vendor damage ratio spread per property (as in D1)
mean_d   = df['mean_dr'].values[idx]
min_d    = tri_a[idx]
max_d    = tri_b[idx]
vendor_d = d_vendor[idx]

fig, ax = plt.subplots(figsize=(10, 3))
ax.vlines(x_pos, min_d, max_d, color=GREY, linewidth=1.2, alpha=0.6,
          label='Vendor range [min, max]')
ax.scatter(x_pos, mean_d,   color=AMBER, s=15, zorder=3, label='Mean across vendors')
ax.scatter(x_pos, vendor_d, color=BLUE,  s=15, zorder=3, label=f'Simulated vendor ({VENDOR_SEED})')
ax.set_xlabel('Properties (sorted by mean damage ratio)')
ax.set_ylabel('Damage ratio  (200yr RP)')
ax.set_xticks([])
ax.legend(frameon=False, fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SINGLE-VENDOR TRANSMISSION  (uses VENDOR_SEED from parameters)
# =============================================================================
framing_label = ('Conditional (p=1)' if FRAMING == 'conditional'
                 else f'Unconditional (p={P_FLOOD_ANNUAL:.4f})')

d_eff  = d_vendor * (P_FLOOD_ANNUAL if FRAMING == 'unconditional' else 1.0)
pd_v   = np.clip(pd0 * np.exp(ALPHA * d_eff), 0, 0.9999)
lgd_v  = np.minimum(lgd0 + LAMBDA * d_eff, 1.0)
el_v   = ead * pd_v * lgd_v         # per-loan expected loss

Loss_v   = el_v.sum()
Capital_v = Capital_0 - Loss_v
CET1_v   = Capital_v / RWA_0
breach_v = CET1_v < CET1_THRESHOLD

print(f"Framing: {framing_label}  |  Vendor seed: {VENDOR_SEED}")
print(f"  Portfolio EL (stressed): GBP{Loss_v:.4f}m  (baseline: GBP{EL_0:.4f}m)")
print(f"  EL uplift:               GBP{Loss_v - EL_0:.4f}m  ({(Loss_v/EL_0 - 1)*100:+.1f}%)")
print(f"  Capital after loss:      GBP{Capital_v:.4f}m")
print(f"  CET1 ratio:              {CET1_v*100:.3f}%  (start: {CET1_START*100:.1f}%,  threshold: {CET1_THRESHOLD*100:.1f}%)")
print(f"  Breach:                  {'YES' if breach_v else 'no'}  |  Buffer: {(CET1_v - CET1_THRESHOLD)*1e4:.0f} bps")

**Figure 1 — Damage ratio vs. loan-level expected loss (single vendor).** Each point is one loan; x-axis is the vendor's damage ratio for that property and y-axis is the resulting stressed EL in GBP thousands. Loans coloured red have a stressed EL above their baseline; blue loans are unchanged (zero or negligible damage). The panel communicates which part of the portfolio drives the capital depletion and whether high-loss loans cluster at high-damage properties.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colours_1 = np.where(el_v > el0, RED, BLUE)
ax.scatter(d_eff, el_v * 1e3, c=colours_1, s=28, alpha=0.75, edgecolors='none', zorder=3)
ax.axvline(0, color=GREY, lw=0.8, ls=':')
ax.set_xlabel('Effective damage ratio  d_eff')
ax.set_ylabel('Stressed EL per loan  (GBP thousands)')
ax.set_title(
    f'Vendor {VENDOR_SEED}  |  {framing_label}  |  '
    f'Portfolio CET1 = {CET1_v*100:.3f}%  '
    f'({"BREACH" if breach_v else "no breach"})'
)
ax.legend(handles=[
    mpatches.Patch(color=RED,  label=f'EL above baseline ({(el_v > el0).sum()})'),
    mpatches.Patch(color=BLUE, label=f'EL at / below baseline ({(el_v <= el0).sum()})'),
], frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# VENDOR UNCERTAINTY SIMULATION  (N_DRAWS draws from triangular distributions)
# =============================================================================
_rng_n = np.random.default_rng(0)

# Vectorised triangular draws: shape (N_DRAWS, n)
u_mat  = _rng_n.uniform(size=(N_DRAWS, n))
span   = np.where(tri_b > tri_a, tri_b - tri_a, 1.0)
fc     = np.where(tri_b > tri_a, (tri_c - tri_a) / span, 0.0)
_left  = tri_a + np.sqrt(np.maximum(u_mat,       0) * span * np.maximum(tri_c - tri_a, 0))
_right = tri_b - np.sqrt(np.maximum(1.0 - u_mat, 0) * span * np.maximum(tri_b - tri_c, 0))
d_mat  = np.where(u_mat <= fc, _left, _right)
d_mat  = np.where(tri_b > tri_a, np.clip(d_mat, 0, 1), tri_a)

# Apply probability framing
d_eff_mat = d_mat * (P_FLOOD_ANNUAL if FRAMING == 'unconditional' else 1.0)

# Transmission: (N_DRAWS, n)
pd_mat  = np.clip(pd0[None, :] * np.exp(ALPHA * d_eff_mat), 0, 0.9999)
lgd_mat = np.minimum(lgd0[None, :] + LAMBDA * d_eff_mat, 1.0)
el_mat  = ead[None, :] * pd_mat * lgd_mat

# Portfolio-level outcomes: (N_DRAWS,)
loss_draws  = el_mat.sum(axis=1)
cet1_draws  = (Capital_0 - loss_draws) / RWA_0
breach_draws = cet1_draws < CET1_THRESHOLD

p_breach = breach_draws.mean()
print(f"N_DRAWS = {N_DRAWS}  |  {framing_label}")
print(f"  Breach probability:        {p_breach:.2%}")
print(f"  CET1 mean:                 {cet1_draws.mean()*100:.3f}%")
print(f"  CET1 std:                  {cet1_draws.std()*100:.3f}%")
print(f"  CET1 P5:                   {np.percentile(cet1_draws, 5)*100:.3f}%")
print(f"  CET1 P50:                  {np.percentile(cet1_draws, 50)*100:.3f}%")
print(f"  CET1 P95:                  {np.percentile(cet1_draws, 95)*100:.3f}%")
print(f"  Loss mean:                 GBP{loss_draws.mean():.4f}m")
print(f"  Loss P95:                  GBP{np.percentile(loss_draws, 95):.4f}m")

**Figure 2 — CET1 ratio distribution across vendor draws.** Each draw represents a bank using a different commercial vendor's flood damage estimates for the same portfolio. The histogram shows how widely the estimated CET1 ratio varies across the vendor population. The dashed red line marks the regulatory minimum; bars left of this line represent vendors whose damage estimates would — if taken at face value — indicate a capital breach.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins_c = np.linspace(cet1_draws.min() * 100, cet1_draws.max() * 100, 60)
bar_colours_c = [RED if c < CET1_THRESHOLD else BLUE
                 for c in (0.5 * (bins_c[:-1] + bins_c[1:]) / 100)]
counts_c, _, patches_c = ax.hist(cet1_draws * 100, bins=bins_c,
                                  edgecolor='white', linewidth=0.3)
for patch, col in zip(patches_c, bar_colours_c):
    patch.set_facecolor(col)
    patch.set_alpha(0.8)
ax.axvline(CET1_THRESHOLD * 100, color=RED,  lw=1.8, ls='--',
           label=f'Threshold  {CET1_THRESHOLD*100:.1f}%')
ax.axvline(CET1_START * 100,     color=GREY, lw=1.2, ls=':',
           label=f'Baseline  {CET1_START*100:.1f}%')
ax.text(0.98, 0.92,
        f'Breach probability
{p_breach:.1%}',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color=RED,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=RED, alpha=0.8))
ax.set_xlabel('CET1 ratio  (%)')
ax.set_ylabel(f'Count  (out of {N_DRAWS} vendor draws)')
ax.set_title(f'CET1 distribution across vendor draws  |  {framing_label}')
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**Figure 3 — Portfolio loss vs. CET1 ratio (scatter).** Each point is one vendor draw. The near-perfect linear relationship (CET1_v = CET1_start − Loss / RWA_0) illustrates the direct mechanical link between climate-induced portfolio losses and capital adequacy. The horizontal dashed line at the regulatory threshold shows the maximum tolerable loss before breach; points above are solvent, points below are in breach.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colours_3 = np.where(breach_draws, RED, BLUE)
ax.scatter(loss_draws, cet1_draws * 100, c=colours_3, s=18, alpha=0.55,
           edgecolors='none', zorder=3)
ax.axhline(CET1_THRESHOLD * 100, color=RED, lw=1.6, ls='--',
           label=f'Threshold  {CET1_THRESHOLD*100:.1f}%')
ax.axvline(max_loss_before_breach, color=RED, lw=1.0, ls=':',
           label=f'Max loss before breach  GBP{max_loss_before_breach:.3f}m')
ax.set_xlabel('Portfolio expected loss  (GBPm)')
ax.set_ylabel('CET1 ratio  (%)')
ax.set_title(f'Loss vs. CET1  |  {framing_label}')
ax.legend(handles=[
    mpatches.Patch(color=RED,  label=f'Breach ({breach_draws.sum()} draws)'),
    mpatches.Patch(color=BLUE, label=f'No breach ({(~breach_draws).sum()} draws)'),
    plt.Line2D([0],[0], color=RED, ls='--', label=f'Threshold {CET1_THRESHOLD*100:.1f}%'),
], frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**Figure 4 — Capital cliff proximity.** The x-axis shows each vendor draw's CET1 buffer above (positive) or below (negative) the regulatory minimum. The shaded red region to the left of zero represents capital breaches. The distribution of buffers communicates not just whether a breach occurs but how *close* the bank is to the cliff across the vendor population — a wide spread near zero indicates high sensitivity to vendor model choice.

In [ ]:
buffer_draws = (cet1_draws - CET1_THRESHOLD) * 100   # percentage points
median_buf = np.median(buffer_draws)
p5_buf     = np.percentile(buffer_draws, 5)

fig, ax = plt.subplots(figsize=(9, 4))
bins_b = np.linspace(buffer_draws.min(), buffer_draws.max(), 60)
ax.hist(buffer_draws, bins=bins_b, color=BLUE, alpha=0.75, edgecolor='white', linewidth=0.3)
# Shade breach region
xlim = ax.get_xlim()
ax.axvspan(buffer_draws.min() - 1, 0, alpha=0.15, color=RED, zorder=0)
ax.axvline(0,          color=RED,  lw=1.8, ls='--', label='Breach boundary (0 pp)')
ax.axvline(median_buf, color=AMBER, lw=1.4, ls='--', label=f'Median buffer  {median_buf:.2f} pp')
ax.axvline(p5_buf,     color=GREY,  lw=1.2, ls=':',  label=f'P5 buffer  {p5_buf:.2f} pp')
ax.set_xlabel('CET1 buffer above threshold  (percentage points)')
ax.set_ylabel(f'Count  (out of {N_DRAWS} vendor draws)')
ax.set_title(f'Capital cliff proximity  |  {framing_label}')
ax.legend(frameon=False, fontsize=9)
ax.text(0.02, 0.93,
        f'{p_breach:.1%} vendors
produce breach',
        transform=ax.transAxes, ha='left', va='top',
        fontsize=9, color=RED,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=RED, alpha=0.7))
plt.tight_layout()
plt.show()

**Figure 5 — Vendor A vs. Vendor B: loan-level loss comparison.** The left panel shows per-loan stressed EL for two specific vendors (sorted by absolute disagreement), highlighting which loans drive the difference. The right panel translates these losses to CET1 ratios and annotates whether each vendor triggers a breach, making the stake of vendor choice concrete.

In [ ]:
# Vendor A and B draws
d_A = triangular_sample(np.random.default_rng(VENDOR_A), tri_a, tri_b, tri_c)
d_B = triangular_sample(np.random.default_rng(VENDOR_B), tri_a, tri_b, tri_c)

def _transmit_single(d_raw):
    d_e   = d_raw * (P_FLOOD_ANNUAL if FRAMING == 'unconditional' else 1.0)
    pd_v  = np.clip(pd0 * np.exp(ALPHA * d_e), 0, 0.9999)
    lgd_v = np.minimum(lgd0 + LAMBDA * d_e, 1.0)
    el    = ead * pd_v * lgd_v
    loss  = el.sum()
    cet1  = (Capital_0 - loss) / RWA_0
    return el, loss, cet1

el_A, Loss_A, CET1_A = _transmit_single(d_A)
el_B, Loss_B, CET1_B = _transmit_single(d_B)
breach_A = CET1_A < CET1_THRESHOLD
breach_B = CET1_B < CET1_THRESHOLD

# Sort loans by absolute disagreement in EL
order_ab = np.argsort(np.abs(el_A - el_B))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Left: per-loan EL comparison ─────────────────────────────────────────────
ax = axes[0]
x_ab = np.arange(n)
w = 0.38
ax.bar(x_ab - w/2, el_A[order_ab] * 1e3, width=w, color=BLUE, alpha=0.8,
       label=f'Vendor {VENDOR_A}')
ax.bar(x_ab + w/2, el_B[order_ab] * 1e3, width=w, color=RED,  alpha=0.8,
       label=f'Vendor {VENDOR_B}')
ax.set_xlabel('Loans (sorted by |ΔEL|, largest first)')
ax.set_ylabel('Stressed EL  (GBP thousands)')
ax.set_xticks([])
ax.legend(frameon=False, fontsize=9)
ax.set_title('Per-loan EL: Vendor A vs. B')

# ── Right: CET1 ratio comparison ─────────────────────────────────────────────
ax = axes[1]
bar_data   = [CET1_A * 100, CET1_B * 100, CET1_START * 100]
bar_labels = [f'Vendor {VENDOR_A}', f'Vendor {VENDOR_B}', 'Baseline']
bar_cols   = [RED if breach_A else BLUE, RED if breach_B else BLUE, GREY]
bars = ax.bar(bar_labels, bar_data, color=bar_cols, alpha=0.85, edgecolor='white')
ax.axhline(CET1_THRESHOLD * 100, color=RED, lw=1.6, ls='--',
           label=f'Threshold {CET1_THRESHOLD*100:.1f}%')
for bar, val in zip(bars, bar_data):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'{val:.3f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('CET1 ratio  (%)')
ax.set_title(
    f'Vendor {VENDOR_A}: {"BREACH" if breach_A else "OK"}  |  '
    f'Vendor {VENDOR_B}: {"BREACH" if breach_B else "OK"}  |  '
    f'|ΔCET1| = {abs(CET1_A - CET1_B)*100:.3f} pp'
)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**Figure 6 — Portfolio loss and CET1 across sorted vendor draws.** Vendor draws are sorted by portfolio loss (x-axis). The left y-axis shows total EL; the right y-axis shows the implied CET1 ratio. The light red shaded region marks the breach zone where CET1 falls below the regulatory minimum. This figure makes the size of the tail that causes breaches immediately visible and communicates the range of outcomes a bank faces depending on which vendor it consults.

In [ ]:
order_loss = np.argsort(loss_draws)
loss_sorted = loss_draws[order_loss]
cet1_sorted = cet1_draws[order_loss]
x_draws = np.arange(N_DRAWS)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

# Shade breach region on the left axis
breach_indices = np.where(cet1_sorted < CET1_THRESHOLD)[0]
if len(breach_indices) > 0:
    ax1.axvspan(breach_indices[0], N_DRAWS - 1, alpha=0.12, color=RED, zorder=0,
                label=f'Breach zone ({len(breach_indices)} draws)')

ax1.plot(x_draws, loss_sorted, color=BLUE, lw=1.5, label='Portfolio loss (left)')
ax2.plot(x_draws, cet1_sorted * 100, color=AMBER, lw=1.5, label='CET1 ratio (right)')
ax2.axhline(CET1_THRESHOLD * 100, color=RED, lw=1.4, ls='--',
            label=f'Threshold {CET1_THRESHOLD*100:.1f}%')

ax1.set_xlabel(f'Vendor draw index (sorted by portfolio loss, n={N_DRAWS})')
ax1.set_ylabel('Portfolio expected loss  (GBPm)', color=BLUE)
ax2.set_ylabel('CET1 ratio  (%)', color=AMBER)
ax1.tick_params(axis='y', labelcolor=BLUE)
ax2.tick_params(axis='y', labelcolor=AMBER)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=9, loc='upper left')
ax1.set_title(f'Sorted vendor draws: loss and CET1  |  {framing_label}')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================
ead_total = ead.sum()

# ── Section 1: Capital calibration ───────────────────────────────────────────
calib_stats = {
    'Total EAD (GBPm)':                    round(ead_total, 3),
    'RWA₀ (GBPm)':                         round(RWA_0, 3),
    'Capital₀ (GBPm)':                     round(Capital_0, 4),
    'Baseline CET1 (%)':                   round(CET1_START * 100, 2),
    'Baseline EL (GBPm)':                  round(EL_0, 4),
    'Buffer above threshold (bps)':        round(buffer_bps, 1),
}
df_calib = pd.DataFrame({'Baseline': calib_stats})

# ── Sections 2 & 3: CET1 dispersion + stability (N_DRAWS simulation) ─────────
cet1_pct   = cet1_draws * 100
buffer_pct = buffer_draws   # already in pp
cond_buffer = buffer_pct[~breach_draws]
loss_cv     = loss_draws.std() / loss_draws.mean()

sim_stats = {
    # CET1 dispersion
    'CET1 mean (%)':                       round(float(np.mean(cet1_pct)), 4),
    'CET1 std (%)':                        round(float(np.std(cet1_pct)), 4),
    'CET1 P5 (%)':                         round(float(np.percentile(cet1_pct,  5)), 4),
    'CET1 P25 (%)':                        round(float(np.percentile(cet1_pct, 25)), 4),
    'CET1 P50 (%)':                        round(float(np.percentile(cet1_pct, 50)), 4),
    'CET1 P75 (%)':                        round(float(np.percentile(cet1_pct, 75)), 4),
    'CET1 P95 (%)':                        round(float(np.percentile(cet1_pct, 95)), 4),
    'CET1 range (P95 − P5, pp)':      round(float(np.percentile(cet1_pct, 95) - np.percentile(cet1_pct, 5)), 4),
    # Capital adequacy stability
    'Breach probability (%)':              round(float(p_breach * 100), 3),
    'Mean CET1 buffer | no breach (pp)':   round(float(cond_buffer.mean()) if len(cond_buffer) > 0 else np.nan, 4),
    'Median loss vs. baseline (GBPm)':     round(float(np.median(loss_draws) - EL_0), 4),
    'Loss CV across vendors':              round(float(loss_cv), 4),
}
df_sim = pd.DataFrame({'N_DRAWS simulation': sim_stats})

# ── Section 4: Pairwise vendor comparison ────────────────────────────────────
if breach_A and breach_B:
    breach_label = 'Both'
elif breach_A:
    breach_label = f'Vendor {VENDOR_A} only'
elif breach_B:
    breach_label = f'Vendor {VENDOR_B} only'
else:
    breach_label = 'Neither'

pair_a = {
    'Portfolio loss (GBPm)':   round(Loss_A, 4),
    'CET1 ratio (%)':          round(CET1_A * 100, 4),
    'Breach':                  'YES' if breach_A else 'no',
    '|ΔCET1| (pp)':       round(abs(CET1_A - CET1_B) * 100, 4),
    'Breach outcome':          breach_label,
}
pair_b = {
    'Portfolio loss (GBPm)':   round(Loss_B, 4),
    'CET1 ratio (%)':          round(CET1_B * 100, 4),
    'Breach':                  'YES' if breach_B else 'no',
    '|ΔCET1| (pp)':       round(abs(CET1_A - CET1_B) * 100, 4),
    'Breach outcome':          breach_label,
}
col_a_label = f'Vendor {VENDOR_A}'
col_b_label = f'Vendor {VENDOR_B}'
df_pair = pd.DataFrame({col_a_label: pair_a, col_b_label: pair_b})
diff_col = {}
for k in pair_a:
    va = pd.to_numeric(pair_a[k], errors='coerce')
    vb = pd.to_numeric(pair_b[k], errors='coerce')
    diff_col[k] = round(vb - va, 4) if pd.notna(va) and pd.notna(vb) else ''
df_pair['Diff (B − A)'] = diff_col

# ── Display ───────────────────────────────────────────────────────────────────
_style_kw = [{'selector': 'caption',
              'props': [('font-weight', 'bold'), ('font-size', '1em')]}]

print(f'Framing: {framing_label}  |  N_DRAWS = {N_DRAWS}  |  ALPHA = {ALPHA}  |  LAMBDA = {LAMBDA}')
print()

for section_name, df_s, col_lbl in [
    ('Capital calibration',            df_calib, 'Baseline'),
    ('Vendor uncertainty — CET1 dispersion & stability', df_sim, 'N_DRAWS simulation'),
]:
    display(df_s.style
            .format(lambda v: f'{v:.4g}' if isinstance(v, float) else str(v),
                    subset=[col_lbl])
            .set_caption(section_name)
            .set_table_styles(_style_kw))

display(df_pair.style
        .format(lambda v: f'{v:.4g}' if isinstance(v, float) else (str(v) if pd.notna(v) else ''),
                subset=[col_a_label, col_b_label])
        .format(lambda v: f'{v:+.4g}' if isinstance(v, (int, float)) and pd.notna(v) else str(v),
                subset=['Diff (B − A)'])
        .set_caption(f'Pairwise vendor comparison  (Vendor {VENDOR_A} vs. Vendor {VENDOR_B})')
        .set_table_styles(_style_kw))